# 03 - Customer complaint, end to end (with human approval)

**Scenario.** Kavya Reddy calls the support desk: her **Nexora Aria 12** says *"Moisture detected in USB port"* and
will not charge, although the port looks dry. The executive asks the assistant to troubleshoot **and raise a ticket**.

**Flow:** Customer Support agent (who is she, what does she own) -> Product Manuals agent (what the manual says) and
Ticket Management agent (existing tickets, then a *proposed* `register_ticket`) -> **the graph pauses for approval**
-> the approved ticket is created through the MCP server -> the answer cites the new ticket.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, `JUDGE_OPENAI_API_KEY` and `SERPER_API_KEY`; the vector stores (`make ingest`); the CRM & Support MCP server (`make mcp-server-up`).

Re-running this notebook exercises the server's **duplicate guard** (the existing open ticket is returned).

In [1]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

Coordinator: gpt-5.1 (high) | specialists: gpt-5.1 (low) | judge: gpt-5.5 (pass >= 80)


In [2]:
# Start the multi-agent service: SQLite memory + CRM MCP tools + 5 specialists + Coordinator graph
from agentic_rag.graph import CoordinatorService

service = await CoordinatorService.create(settings)
print(service.crm.describe())
print(service.roster())

CRM MCP server at http://localhost:8765/mcp: customer=['get_customer_profile', 'search_customer_profiles'], ticket_read=['get_tickets_by_customer', 'search_tickets'], ticket_write=['register_ticket', 'resolve_ticket']
- `hr_policy` (available): HR policies (leave, travel and expenses, medical insurance, conduct, POSH, WFH, IT security, privacy, attendance, appraisal, whistleblowing, anti-bribery, notice period). Current versions by default; flags superseded values.
- `product_manuals` (available): TV and phone user manuals: specifications, setup, error codes, troubleshooting steps, warranty and safety. Pins the exact model (near-duplicate manuals).
- `customer_support` (available): Find customers by ID, name, email or phone and report their profile: active status, credit limit, contact details, remarks such as products owned. Read-only.
- `ticket_management` (available): Support ticket history and search (by customer, status, priority, product model, text). Proposes new tickets or reso

In [3]:
QUESTION = ("Kavya Reddy called: her Nexora Aria 12 says 'Moisture detected in USB port' and will not charge, "
            "although the port looks dry. Tell me what the manual recommends and raise a support ticket for her.")

result = await service.ask(QUESTION, user_id="support-exec-ravi", on_event=print)
result

[   0.0s] start_turn      done
[   0.0s] load_memory     3 long-term facts recalled
[  23.8s] plan            Initial plan for this question. -> s1=product_manuals, s2=ticket_management
[  39.1s] run_step        s2 ticket_management done in 15.3s (confidence 0.72, 0 evidence, 1 proposed actions)
[  39.2s] run_step        s1 product_manuals done in 15.4s (confidence 0.86, 2 evidence, 0 proposed actions)
[  59.9s] reflect         synthesize: The specialists have already retrieved the relevant Nexora Aria 12 manual guidance for the moisture warning and drafted 
[  59.9s] __interrupt__   PAUSED for approval: [{'id': 0, 'tool': 'register_ticket', 'arguments': {'title': "Nexora Aria 12 - 'Moisture detected in USB port' error and will not charge", 'description': 'Customer: Kavya Reddy.\nProduct: Nexora Aria 12 smartphone.\n\nIssue summary:\n- Device displays the exact on-screen error message: "Moisture detected in USB port".\n- Phone will not charge when a charger is connected.\n- Customer re

**status** `needs_approval` | **thread** `thread-8a773e56` | **elapsed** 59.9s

```text
plan round 1 (Initial plan for this question.): s1:product_manuals, s2:ticket_management
timeline:
    s1 product_manuals       0.0s ->   15.4s |########################################|
    s2 ticket_management     0.0s ->   15.3s |####################################### |
INTERRUPT: {'type': 'approval', 'actions': [{'id': 0, 'tool': 'register_ticket', 'arguments': {'title': "Nexora Aria 12 - 'Moisture detected in USB port' error and will not charge", 'description': 'Customer: Kavya Reddy.\nProduct: Nexora Aria 12 smartphone.\n\nIssue summary:\n- Device displays the exact on-screen error message: "Moisture detected in USB port".\n- Phone will not charge when a charger is connected.\n- Customer reports that the USB port looks dry on inspection.\n\nImmediate symptom:\n- Device is currently not charging at all; core charging function is unavailable.\n\nTroubleshooting context so far (as reported):\n- Customer performed a basic visual inspection of the USB port and did not see any visible moisture, liquid, or debris.\n- No other troubleshooting steps have been documented yet (e.g., power-cycling, changing charger/cable, or following the manual’s moisture-clearing procedure).\n\nRequest:\n- Provide guidance per the Nexora Aria 12 manual for handling the "Moisture detected in USB port" warning, including safe drying steps and any waiting period before reconnecting power.\n- If the issue persists after recommended steps, assess for possible hardware fault in the charging port or related components and arrange service as needed.', 'priority': 'high', 'product_model': 'Nexora Aria 12'}, 'reason': "Customer reports that her Nexora Aria 12 displays the on-screen error 'Moisture detected in USB port' and the phone will not charge, even though the USB port appears dry on visual inspection. There is no existing open ticket for this product and symptom, and this is a core functionality issue (device cannot charge) requiring timely support.", 'proposed_in': 's2'}], 'how_to_answer': "'approve' or 'reject' for all, or {'decisions': [{'id': 0, 'decision': 'approve'|'reject'|'edit', 'arguments': {...}}]}"}
```

**Waiting for you:** `{'type': 'approval', 'actions': [{'id': 0, 'tool': 'register_ticket', 'arguments': {'title': "Nexora Aria 12 - 'Moisture detected in USB port' error and will not charge", 'description': 'Customer: Kavya Reddy.\nProduct: Nexora Aria 12 smartphone.\n\nIssue summary:\n- Device displays the exact on-screen error message: "Moisture detected in USB port".\n- Phone will not charge when a charger is connected.\n- Customer reports that the USB port looks dry on inspection.\n\nImmediate symptom:\n- Device is currently not charging at all; core charging function is unavailable.\n\nTroubleshooting context so far (as reported):\n- Customer performed a basic visual inspection of the USB port and did not see any visible moisture, liquid, or debris.\n- No other troubleshooting steps have been documented yet (e.g., power-cycling, changing charger/cable, or following the manual’s moisture-clearing procedure).\n\nRequest:\n- Provide guidance per the Nexora Aria 12 manual for handling the "Moisture detected in USB port" warning, including safe drying steps and any waiting period before reconnecting power.\n- If the issue persists after recommended steps, assess for possible hardware fault in the charging port or related components and arrange service as needed.', 'priority': 'high', 'product_model': 'Nexora Aria 12'}, 'reason': "Customer reports that her Nexora Aria 12 displays the on-screen error 'Moisture detected in USB port' and the phone will not charge, even though the USB port appears dry on visual inspection. There is no existing open ticket for this product and symptom, and this is a core functionality issue (device cannot charge) requiring timely support.", 'proposed_in': 's2'}], 'how_to_answer': "'approve' or 'reject' for all, or {'decisions': [{'id': 0, 'decision': 'approve'|'reject'|'edit', 'arguments': {...}}]}"}`

## The approval interrupt
The Ticket Management agent cannot write. It *proposed* an action; the graph stopped in `approve_actions` and saved its state in the SQLite checkpointer. Inspect the proposal:

In [4]:
import json
print(result.status)
print(json.dumps(result.interrupt, indent=2)[:3000])

needs_approval
{
  "type": "approval",
  "actions": [
    {
      "id": 0,
      "tool": "register_ticket",
      "arguments": {
        "title": "Nexora Aria 12 - 'Moisture detected in USB port' error and will not charge",
        "description": "Customer: Kavya Reddy.\nProduct: Nexora Aria 12 smartphone.\n\nIssue summary:\n- Device displays the exact on-screen error message: \"Moisture detected in USB port\".\n- Phone will not charge when a charger is connected.\n- Customer reports that the USB port looks dry on inspection.\n\nImmediate symptom:\n- Device is currently not charging at all; core charging function is unavailable.\n\nTroubleshooting context so far (as reported):\n- Customer performed a basic visual inspection of the USB port and did not see any visible moisture, liquid, or debris.\n- No other troubleshooting steps have been documented yet (e.g., power-cycling, changing charger/cable, or following the manual\u2019s moisture-clearing procedure).\n\nRequest:\n- Provide guid

Approve it (try `"reject"` instead to see the other path). The graph resumes from the checkpoint, executes the MCP `register_ticket` tool, writes the answer and runs the judge.

In [5]:
DECISION = "approve"
if result.status == "needs_approval":
    result = await service.resume(result.thread_id, DECISION, on_event=print)
result

[   0.0s] approve_actions done
[   0.3s] execute_actions Executed register_ticket: 1 validation error for call[register_ticket]
customer_id
  Missing required argument [type=missing_argument, input_value={'t
[  50.1s] synthesize      done
[  52.7s] judge           attempt 1: 71.0/100 FAIL (ground 55, rel 80, compl 75, ctx 85)
[ 106.3s] plan            RETRY after the quality review failed. Previous answer: -> no new steps
[ 137.9s] synthesize      done
[ 141.1s] judge           attempt 2: 93.2/100 PASS (ground 98, rel 90, compl 88, ctx 95)
[ 141.1s] finalize        done
[ 143.9s] save_memory     done


**status** `completed` | **thread** `thread-8a773e56` | **elapsed** 143.9s  
**judge** 93.2/100 | **passed** True | **attempts** 2 | **specialist runs** 2

```text
plan round 1 (Initial plan for this question.): s1:product_manuals, s2:ticket_management
plan round 2 (RETRY after the quality review failed. Previous answer:): no steps
timeline:
    s1 product_manuals       0.0s ->   15.4s |########################################|
    s2 ticket_management     0.0s ->   15.3s |####################################### |
judge attempt 1: 71.0/100 FAIL (ground 55, rel 80, compl 75, ctx 85)
judge attempt 2: 93.2/100 PASS (ground 98, rel 90, compl 88, ctx 95)
```

---

Here’s what the Nexora Aria 12 manual recommends, and what happened with the support ticket request.

**Manual guidance for “Moisture detected in USB port”**

- The message means the phone has detected liquid or sweat in the charging port. [E1]  
- Immediate step: **disconnect/unplug the USB cable** from the phone. [E1]  
- Then **let the USB port dry fully** before trying to charge again. [E1]  
- Under its water‑resistance guidance, the manual adds: **do not charge the phone when the USB port is wet; if it is wet, dry it for at least 30 minutes before charging.** [E2]  
- It also warns that **liquid damage is not covered by the warranty**, regardless of the phone’s IP rating. [E2]  

The manual does not give any extra drying techniques (like cloth, rice, or heat) nor any override method if the port appears dry but the warning and charging block continue; it only says to keep the port dry and avoid charging until it is fully dry. [E1][E2]

**Support ticket status**

- A new support ticket was attempted for this “Moisture detected in USB port” / not‑charging issue, but **the ticket system rejected the request because a required `customer_id` field was missing**, so **no new ticket could be created via this channel.** [E3]  

To get a ticket opened, she’ll need to go through a support channel that can attach the issue to her customer account (so the missing `customer_id` can be supplied).

In [7]:
# DECISION = "approve"
# if result.status == "needs_approval":
QUESTION = "My customer id : CUST-0003"
result = await service.ask(QUESTION, user_id="support-exec-ravi", on_event=print)
result

[   0.0s] start_turn      done
[   0.0s] load_memory     4 long-term facts recalled
[   9.9s] plan            Initial plan for this question. -> no new steps
[  34.3s] synthesize      done
[  37.7s] judge           attempt 1: 18.0/100 FAIL (ground 0, rel 40, compl 30, ctx 0)
[  42.5s] plan            RETRY after the quality review failed. Previous answer: -> s1=customer_support, s2=ticket_management
[  48.6s] run_step        s1 customer_support done in 6.1s (confidence 1.00, 1 evidence, 0 proposed actions)
[  49.0s] run_step        s2 ticket_management done in 6.5s (confidence 0.97, 1 evidence, 0 proposed actions)
[  52.0s] reflect         synthesize: The existing specialist reports fully answer the implied request: to summarize the CRM profile and ticket status for cus
[  59.5s] synthesize      done
[  62.8s] judge           attempt 2: 93.2/100 PASS (ground 95, rel 90, compl 90, ctx 100)
[  62.8s] finalize        done
[  64.2s] save_memory     done


**status** `completed` | **thread** `thread-f69fbf99` | **elapsed** 64.2s  
**judge** 93.2/100 | **passed** True | **attempts** 2 | **specialist runs** 2

```text
plan round 1 (Initial plan for this question.): no steps
plan round 2 (RETRY after the quality review failed. Previous answer:): s1:customer_support, s2:ticket_management
timeline:
    s1 customer_support      0.0s ->    6.1s |#####################################   |
    s2 ticket_management     0.0s ->    6.5s |########################################|
judge attempt 1: 18.0/100 FAIL (ground 0, rel 40, compl 30, ctx 0)
judge attempt 2: 93.2/100 PASS (ground 95, rel 90, compl 90, ctx 100)
```

---

You’re registered as **customer CUST-0003: Kavya Reddy**, with a **Nexora Aria 12 (eSIM user)** in our system. [E1]  
Currently, there are **no open support tickets** on your account; your last ticket (**TKT-000083** about eSIM activation on the Aria 12) is marked **resolved**. [E2]

Tell me what you’d like help with now (e.g., a new issue with your phone, billing, account details), and I’ll assist you step by step.

## Verify in the CRM through the MCP tools

In [6]:
from agentic_rag.mcp_client import tool_text

tool = next(t for t in service.crm.ticket_read if t.name == "get_tickets_by_customer")
print(tool_text(await tool.ainvoke({"customer_id": "CUST-0003", "status": "open"}))[:1500])

{"total":0,"count":0,"tickets":[]}


In [8]:
# Close the memory databases
await service.aclose()

## What to observe
- `depends_on`: the product and ticket steps wait for the customer lookup, then run in parallel (timeline).
- Writes need a human: the interrupt payload shows tool, arguments and reason.
- The executed action becomes evidence (`kind="action"`) that the final answer cites.